# FINAL MODEL — Sinhala token-level offensive detection

Phase 2, Step 8. Full-train refit with the subword channel.

Trains **two** models and scores each on test once:

| model | channels | trainable | needs fastText? |
|---|---|---|---|
| **full** | word + subword | ~330k | yes |
| **minimal** | subword only | ~176k | **no** |

Everything is frozen from the Step 7b ablations. This notebook changes nothing;
it just runs the chosen configuration on all 7,500 training tweets.

---
## Colab setup — do this first

1. **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**
2. Push your latest code to GitHub, then edit `REPO` in cell 3 below
3. Run every cell top to bottom

Expect **60–90 minutes** total on a T4. Colab wipes the session when it ends,
so download your results at the end (last cell). Keep the tab open or Colab
will disconnect you as idle.


## 1. Confirm the GPU


In [28]:
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU'


GPU 0: Tesla T4 (UUID: GPU-e1b0c9ce-3497-a454-39c3-a5f5c3368828)
torch 2.11.0+cu128 | CUDA True


## 2. Install packages

`datasets` is pinned below 3.0 because SOLD is a 2022-era dataset and newer
versions dropped support for its loading script.


In [29]:
!pip install -q 'datasets<3.0.0' pytorch-crf sentencepiece 2>&1 | tail -1
from torchcrf import CRF
import sentencepiece
print('ready')


ready


## 3. Get the code

**Edit REPO.** If the repo is private, skip this and use the upload cell below.


In [30]:
REPO = 'https://github.com/hatheem-r/project_DNN.git'   # <-- EDIT THIS

import os, shutil
if os.path.exists('project'): shutil.rmtree('project')
!git clone -q $REPO project
%cd project
!ls src/ notebooks/ tests/


/content/project/project/project/project
notebooks/:
01_data_exploration.ipynb  03_embeddings.py	   06_subword_tokenizer.py
01_data_exploration.py	   04_baseline.py	   07_subword_model.py
02_metric_check.py	   05_full_train_refit.py  08_final_model.py

src/:
data.py     embeddings.py  metrics.py  models	subword.py
dataset.py  __init__.py    model.py    seed.py	train.py

tests/:
test_metrics.py  test_subword_alignment.py


*Private repo alternative: zip `src/`, `notebooks/` and `tests/`, then run this.*


In [31]:
# from google.colab import files
# import zipfile, os
# up = files.upload()
# os.makedirs('project', exist_ok=True)
# zipfile.ZipFile(list(up)[0]).extractall('project')
# %cd project
# !ls


## 4. Safety checks — run BEFORE training

The alignment tests catch the one bug that would silently ruin everything:
labels are one per word, subword pieces are smaller than words, and if the
piece tensor ever has the wrong number of word-rows the labels shift against
the words. The model still trains, the loss still falls, and the score is
quietly wrong with nothing crashing.

**If either suite fails, stop. Do not train.**


In [32]:
!python tests/test_metrics.py
!python tests/test_subword_alignment.py


1 perfect prediction               OK
2 all-negative collapse detected   OK
3 hand-computed P/R/F1             OK
4 orientation not swapped          OK
5 pooled not per-sentence          OK
6 macro F1 != offensive F1         OK
7 padding / truncation             OK
8 empty positive class safe        OK
9 seed aggregation                 OK

all 9 tests passed

1. Dataset produces one piece-list per word
  every tweet: len(pieces) == len(words) == len(labels)      OK

2. Collate preserves the invariant after padding
  piece block (3, 5, 3) matches word block (3, 5)            OK
  real words have >=1 piece; padded words have 0             OK

3. Encoder returns exactly one vector per word
  bilstm pooling: (3, 5, 3) -> (3, 5, 32)                    OK
  mean pooling:   (3, 5, 3) -> (3, 5, 32)                    OK

4. Padded word positions produce all-zero vectors
  no signal leaks from padding into the word representation  OK

5. A word's vector depends only on ITS OWN pieces
  pieces 

## 5. Download the Sinhala fastText vectors

About 600 MB, two to three minutes. Needed for the **full** model; the
**minimal** model does not use them at all.

Do not unzip — the loader reads `.gz` directly.


In [33]:
!mkdir -p embeddings results artifacts
![ -f embeddings/cc.si.300.vec.gz ] || wget -q --show-progress -O embeddings/cc.si.300.vec.gz https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.si.300.vec.gz
!ls -lh embeddings/


embeddings/cc.si.30 100%[===================>] 459.17M   198MB/s    in 2.3s    
total 460M
-rw-r--r-- 1 root root 460M Jan 18  2019 cc.si.300.vec.gz


## 6. Sanity check — 1 seed, 3 epochs

Confirms the pipeline runs on GPU before committing to the full job.
The score will be poor because it barely trains. That is expected — you are
checking that it runs, not what it scores.


In [34]:
!python notebooks/08_final_model.py --model full --seeds 1 --epochs 3 2>&1 | tail -20


Published XLM-R + TSD transfer       0.7300                                   ~560M

CHANGE FROM THE PHASE 1 BASELINE
  0.5965 -> 0.6904   (+0.0939)
  precision 0.7452 -> 0.7114  (-0.0338)
  recall    0.4979 -> 0.6706  (+0.1727)

  The gain is recall-driven at near-constant precision. That is the mechanism
  predicted from the measured 48.7% unseen-type rate, not a threshold shift.
  Say this in the paper - the mechanism is what makes the number credible.
EFFICIENCY
  ours 329,658 trainable vs XLM-R-large ~560,000,000
  = 1,699x fewer trainable parameters

rows written to results/results_final.csv

WRITE IT UP HONESTLY. The headline is not "we nearly match XLM-R". It is:
a lightweight model with a subword channel reaches this F1 at a small fraction
of the parameters, and the gain is precision-neutral and recall-driven.
The mechanism is what makes it credible.


## 7. THE FINAL RUN

Both models, 5 seeds each, 35 fixed epochs, test scored once.

The epoch budget of 35 is the median best-epoch from the bpe_1000 validation
runs (30, 30, 41, 35, 37). It came from validation, never from test — which is
what makes a fixed budget legitimate here.

**Do not re-run this with different settings if you dislike the number.** That
would be tuning on test. Whatever it gives is the result.


In [35]:
!python notebooks/08_final_model.py --both 2>&1 | tee results/step8_final.txt



0. SETUP
device cuda
train 7,500 (FULL official split)   test 2,500
No validation split. Fixed epoch budget instead of early stopping.
tokenizer: 1,000 pieces
word vocabulary: 33,006 (was 28,456 from train-part)
  scanned 808,044 lines, matched 26,668 of 33,006 vocab words
embedding matrix (33006, 300), real vectors for 80.8%

frozen configuration
  tokenizer       bpe 1000 (1,000 pieces), trained on full train
  piece_dim       50
  subword_dim     100
  pooling         bilstm
  hidden          64
  dropout         0.5
  lr              0.001
  batch           32
  crf             True
  epochs          35 FIXED, no early stopping
  seeds           [1, 2, 3, 4, 5]

1. TRAIN

MODEL: FULL   (word + subword)
parameters:
  total                        10,231,458
  trainable                       329,658
  frozen                        9,901,800
  word_embedding                9,901,800
  subword_channel                  90,800
  lstm_input_dim                      400
  non_embedding_tra

## 8. Optional: the last owed ablation

`--pooling mean` replaces the per-word BiLSTM with a simple average. It answers
whether piece **order** matters, and it may be considerably faster since the
subword encoder is now the bottleneck.

This is an ablation row, not a candidate for the headline model.


In [36]:
!python notebooks/08_final_model.py --model full --pooling mean 2>&1 | tail -25


Published SinBERT                    0.6200                                   ~110M
Our word list                        0.6521            0.6361   0.6689            0
Published XLM-T                      0.7000              0.64     0.77        ~270M
OURS: word + subword                 0.6822   0.0092   0.7931   0.5991      293,958
Published XLM-R                      0.7200              0.68     0.76        ~560M
Published XLM-R + TSD transfer       0.7300                                   ~560M

CHANGE FROM THE PHASE 1 BASELINE
  0.5965 -> 0.6822   (+0.0857)
  precision 0.7452 -> 0.7931  (+0.0479)
  recall    0.4979 -> 0.5991  (+0.1012)

  The gain is recall-driven at near-constant precision. That is the mechanism
  predicted from the measured 48.7% unseen-type rate, not a threshold shift.
  Say this in the paper - the mechanism is what makes the number credible.
EFFICIENCY
  ours 293,958 trainable vs XLM-R-large ~560,000,000
  = 1,905x fewer trainable parameters

rows written to r

## 9. Download everything

Colab deletes the session when it closes. Save these and commit them.


In [1]:
from google.colab import files
import os
for f in ['results/results_final.csv', 'results/step8_final.txt']:
    if os.path.exists(f):
        files.download(f)
    else:
        print('missing:', f)


ModuleNotFoundError: No module named 'google.colab'

---
## After it finishes

**Record in the README:** both models' F1 with standard deviations, precision
and recall, trainable parameter counts, and the change from the Phase 1
baseline of 0.5965.

**Check the precision/recall pattern.** The script prints a different message
depending on what it finds. If recall rose while precision held, the
unseen-morphology mechanism is confirmed on the full training split and you can
claim it. If precision fell instead, the model is simply firing more freely and
you must not claim that mechanism.

**Write it up honestly.** The headline is not "we nearly match XLM-R". It is:
*a lightweight model with a subword channel reaches this F1 at a small fraction
of the parameters, and the gain is precision-neutral and recall-driven.* The
mechanism is what makes a reviewer believe the number.

**Then Piece 2:** the joint sentence + token head.
